### Fine Tuning du Roberta

In [1]:
 pip install pandas numpy scikit-learn transformers accelerate tqdm matplotlib seaborn paramiko tiktoken sentencepiece protobuf


[notice] A new release of pip is available: 23.3.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import random
import paramiko
import tiktoken
import sentencepiece
import warnings
warnings.filterwarnings('ignore') 

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizer
from transformers import RobertaForMultipleChoice
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

from tqdm.auto import tqdm 
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Appareil utilisé pour l'entraînement : {device}")

if torch.cuda.is_available():
    print(f"Modèle du GPU : {torch.cuda.get_device_name(0)}")

Appareil utilisé pour l'entraînement : cuda
Modèle du GPU : NVIDIA GeForce RTX 3090


#### 1. Préparation des données

Avant de pouvoir entraîner notre modèle RoBERTa, nous avons dû concevoir une pipeline de préparation des données stricte. Il faut savoir que nous avons fait le choix d'une architecture en RobertaForMultipleChoice. En effet, une limite du Roberta classique est le nombre de tokens qui est limité à 512. Or, les commentaires de CMV sont longs donc afin de pouvoir au mieux ces tokens, au lieu de donner à notre modèle la concaténation de [OP + Argument 0 + Argument 1], on lui donne la paire [OP + Argument 0] et la paire [OP + Argument 1]. Ainsi, chaque paire bénéficie de 512 tokens.

Concrètement, dans cette partie, on reformate le dataset pour pouvoir passer à notre modèle les pairs. L'extraction se fait grâce à pair_id. On fait en sorte que l'ordre dans lequel les paires sont passées dépende du hasard afin de réduire le biais de position.

Concernant la taille du batch, disposant des GPU de Telecoms, nous avons fait le choix d'utiliser une taille de 8 qui permet d'éviter de rester bloquer dans des minimas locaux.

In [8]:
def format_cmv_data(raw_df):
    print("Reformatage des données en cours...")
    # Séparation des gagnants et perdants
    winners_df = raw_df[raw_df['success'] == 1].set_index('pair_id')
    losers_df = raw_df[raw_df['success'] == 0].set_index('pair_id')
    
    # Intersection pour ne garder que les paires complètes
    valid_pairs = winners_df.index.intersection(losers_df.index)
    
    formatted_data = []
    
    for p_id in valid_pairs:
        op = winners_df.loc[p_id, 'op_text']
        winning_arg = winners_df.loc[p_id, 'text']
        losing_arg = losers_df.loc[p_id, 'text']
        
        # Sécurité si doublons d'index
        if isinstance(op, pd.Series): op = op.iloc[0]
        if isinstance(winning_arg, pd.Series): winning_arg = winning_arg.iloc[0]
        if isinstance(losing_arg, pd.Series): losing_arg = losing_arg.iloc[0]
        
        # Randomisation pour éviter le biais de position
        if random.random() > 0.5:
            arg_0, arg_1 = winning_arg, losing_arg
            label = 0
        else:
            arg_0, arg_1 = losing_arg, winning_arg
            label = 1
            
        formatted_data.append({
            'pair_id': p_id,
            'op': str(op),
            'arg_0': str(arg_0),
            'arg_1': str(arg_1),
            'label': label
        })
        
    print(f"Extraction réussie : {len(formatted_data)} paires prêtes.")
    return formatted_data

In [9]:
class CMVDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Tokenisation de l'Option 0 : [CLS] OP [SEP] Arg_0 [SEP]
        enc_0 = self.tokenizer(
            item['op'], item['arg_0'], 
            truncation=True, max_length=self.max_length, 
            padding='max_length', return_tensors='pt'
        )
        
        # Tokenisation de l'Option 1 : [CLS] OP [SEP] Arg_1 [SEP]
        enc_1 = self.tokenizer(
            item['op'], item['arg_1'], 
            truncation=True, max_length=self.max_length, 
            padding='max_length', return_tensors='pt'
        )
        
        # RoBERTa Multiple Choice attend une dimension (num_choices, seq_length)
        return {
            'input_ids': torch.cat([enc_0['input_ids'], enc_1['input_ids']], dim=0),
            'attention_mask': torch.cat([enc_0['attention_mask'], enc_1['attention_mask']], dim=0),
            'labels': torch.tensor(item['label'], dtype=torch.long)
        }

In [10]:
df = pd.read_csv('/home/infres/ydahasse-24/WAC.csv')

In [11]:
processed_data = format_cmv_data(df)

Reformatage des données en cours...
Extraction réussie : 3866 paires prêtes.


In [12]:
train_data, temp_data = train_test_split(processed_data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"Répartition : {len(train_data)} Train | {len(val_data)} Val | {len(test_data)} Test")

Répartition : 3092 Train | 387 Val | 387 Test


In [13]:
print("Chargement du Tokenizer RoBERTa...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

Chargement du Tokenizer RoBERTa...


In [14]:
train_dataset = CMVDataset(train_data, tokenizer)
val_dataset = CMVDataset(val_data, tokenizer)
test_dataset = CMVDataset(test_data, tokenizer)

In [15]:
BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

#### Entrainement du modèle

In [16]:
# Hyperparamètres d'entraînement
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 2 # Nombre d'époques à attendre avant l'arrêt précoce si pas d'amélioration

In [31]:
model = RobertaForMultipleChoice.from_pretrained('roberta-base')
model.to(device) 

# Configuration de l'Optimiseur
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
roberta.pooler.dense.bias   | MISSING    | 
classifier.bias             | MISSING    | 
roberta.pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# Configuration du Scheduler 
total_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_steps
)

NameError: name 'optimizer' is not defined

In [35]:
# Variables pour le suivi et l'arrêt précoce
best_val_acc = 0.0
patience_counter = 0

print("Début de l'entraînement...")

for epoch in range(EPOCHS):
    # ========================================
    #               ENTRAÎNEMENT
    # ========================================
    model.train()
    total_train_loss = 0
    
    # Barre de progression pour le lot d'entraînement
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for batch in train_loop:
        optimizer.zero_grad() # Réinitialisation des gradients
        
        # Envoi des données sur le GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass (Prédiction)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()
        
        # Backward pass (Calcul des gradients)
        loss.backward()
        
        # Prévention de l'explosion des gradients (Gradient Clipping)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        # Mise à jour des poids
        optimizer.step()
        scheduler.step()
        
        # Mise à jour de la barre de progression
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)

    # ========================================
    #               VALIDATION
    # ========================================
    model.eval()
    val_preds, val_labels = [], []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
    
    # Pas de calcul de gradient pendant l'évaluation (économise mémoire et temps)
    with torch.no_grad():
        for batch in val_loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits # Scores bruts avant Softmax
            
            # L'option avec le plus grand score est notre prédiction
            preds = torch.argmax(logits, dim=1)
            
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(val_labels, val_preds)
    
    print(f"\n--- Bilan de l'Époque {epoch+1} ---")
    print(f"Perte d'entraînement (Loss) : {avg_train_loss:.4f} | Précision de Validation : {val_acc:.4f}")
    
    # ========================================
    #             ARRÊT PRÉCOCE (Early Stopping)
    # ========================================
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # On sauvegarde le meilleur modèle localement
        torch.save(model.state_dict(), 'best_roberta_cmv.pt')
        print("=> Nouveau record ! Modèle sauvegardé sous 'best_roberta_cmv.pt'")
        patience_counter = 0 # On remet le compteur à zéro
    else:
        patience_counter += 1
        print(f"=> Pas d'amélioration. Patience : {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("\nArrêt précoce déclenché : Le modèle a cessé de s'améliorer. Fin de l'entraînement.")
            break

Début de l'entraînement...


Epoch 1/4 [Train]:   0%|          | 0/387 [00:00<?, ?it/s]

Epoch 1/4 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]


--- Bilan de l'Époque 1 ---
Perte d'entraînement (Loss) : 0.6875 | Précision de Validation : 0.6124
=> Nouveau record ! Modèle sauvegardé sous 'best_roberta_cmv.pt'


Epoch 2/4 [Train]:   0%|          | 0/387 [00:00<?, ?it/s]

...

Epoch 3/4 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]


--- Bilan de l'Époque 3 ---
Perte d'entraînement (Loss) : 0.5864 | Précision de Validation : 0.6408
=> Nouveau record ! Modèle sauvegardé sous 'best_roberta_cmv.pt'


Epoch 4/4 [Train]:   0%|          | 0/387 [00:00<?, ?it/s]

Epoch 4/4 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]


--- Bilan de l'Époque 4 ---
Perte d'entraînement (Loss) : 0.4652 | Précision de Validation : 0.6382
=> Pas d'amélioration. Patience : 1/2


In [18]:
def evaluate_on_test(model, test_loader, device):
    print("Chargement des meilleurs poids sauvegardés...")
    # On s'assure de charger le meilleur modèle (pas forcément celui de la dernière époque)
    model.load_state_dict(torch.load('best_roberta_cmv.pt'))
    model.eval() # Mode évaluation
    
    test_preds = []
    test_labels = []
    
    test_loop = tqdm(test_loader, desc="Évaluation Test")
    
    with torch.no_grad():
        for batch in test_loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            # Récupération des prédictions
            preds = torch.argmax(outputs.logits, dim=1)
            
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())
            
    # Calcul des métriques
    final_accuracy = accuracy_score(test_labels, test_preds)
    final_f1 = f1_score(test_labels, test_preds, average='macro')
    
    print("\n" + "="*40)
    print("RÉSULTATS FINAUX SUR LE JEU DE TEST")
    print("="*40)
    print(f"Précision (Accuracy) : {final_accuracy:.4f}")
    print(f"Score F1 (Macro)     : {final_f1:.4f}")
    print("\nRapport Détaillé :")
    
    # target_names correspondent à nos labels (0 = Arg_0 gagne, 1 = Arg_1 gagne)
    # Grâce à notre randomisation (anti-biais), ces deux classes doivent être équilibrées.
    print(classification_report(test_labels, test_preds, target_names=["Option 0 Gagne", "Option 1 Gagne"]))

# Lancement de l'évaluation
evaluate_on_test(model, test_loader, device)

NameError: name 'model' is not defined

##### Observations

**Performance générale.** Une accuracy de 65.89 % est cohérente avec
les résultats attendus pour RoBERTa-base sur ce dataset. Les travaux
antérieurs (Tan et al., 2016 ; Labruna et al., 2026) situent les
modèles de type BERT fine-tunés entre 63 et 70 % sur le split test
du WA corpus (807 paires), notre split réduit (387 paires) donnant
des résultats comparables.

**Équilibre des classes.** Le support est quasi-parfaitement équilibré
(193 vs 194 exemples), ce qui valide que la tâche est formulée comme
un problème pairwise symétrique. L'accuracy et le F1 macro sont
identiques, confirmant l'absence de biais vers une classe.

**Symétrie precision/recall.** Les scores precision et recall sont
proches pour les deux classes (écart max : 2 points), ce qui indique
que le modèle n'a pas de tendance systématique à sur-prédire l'une
ou l'autre. Les erreurs sont distribuées uniformément.

##### Limites identifiées

- **Fenêtre de 512 tokens.** RoBERTa-base ne peut traiter que 512
  tokens par exemple. Les arguments CMV étant souvent longs, une
  partie du contenu informatif est probablement tronquée, ce qui
  pénalise les cas où les points clés apparaissent en fin d'argument.

- **Capacité du modèle.** RoBERTa-base (125 M paramètres) reste un
  encodeur de taille modeste. La tâche de persuasion nécessite une
  compréhension fine des relations sémantiques et rhétoriques entre
  les arguments, ce qui dépasse les capacités d'un modèle de base.

- **Taille du jeu d'entraînement.** Avec ~2 746 exemples d'entraînement,
  le dataset reste petit pour le fine-tuning d'un transformer, ce qui
  augmente le risque de sur-apprentissage ou de sous-optimisation